# HepatoSwitch Frontiers Analysis Pipeline

This notebook is the executable orchestrator for the HepatoSwitch modular workflow. Place it in the same directory as `config.py`, `geo_ingest.py`, `preprocess.py`, `differential_expression.py`, `pathway_enrichment.py`, `network_switches.py`, `ml_classifier.py`, `figures.py`, and `simulate.py`.

The workflow attempts live GEO retrieval first. If retrieval or preprocessing fails, it explicitly switches to the simulated-data fallback and labels all resulting outputs as simulated.

In [ ]:
import argparse
import os
import traceback
import warnings

import pandas as pd

import config as C
import differential_expression
import figures
import geo_ingest
import ml_classifier
import network_switches
import pathway_enrichment
import preprocess
import simulate

warnings.filterwarnings("ignore")


def banner(text):
    print("\n" + "=" * 74)
    print(f"  {text}")
    print("=" * 74, flush=True)


def load_cached_data():
    """Load cached matrices and report whether they are simulated."""
    pre = {
        "discovery_expr": pd.read_parquet(
            os.path.join(C.RES_DIR, "discovery_expr.parquet")
        ),
        "discovery_meta": pd.read_csv(
            os.path.join(C.RES_DIR, "discovery_meta.csv"), index_col=0
        ),
        "validation_expr": pd.read_parquet(
            os.path.join(C.RES_DIR, "validation_expr.parquet")
        ),
        "validation_meta": pd.read_csv(
            os.path.join(C.RES_DIR, "validation_meta.csv"), index_col=0
        ),
    }
    simulated = (
        "platform" in pre["discovery_meta"].columns
        and pre["discovery_meta"]["platform"].eq("simulated").any()
    )
    return pre, simulated


def get_data(args):
    """Return the preprocessed data dictionary and simulation-status flag."""
    cache_files = [
        "discovery_expr.parquet",
        "discovery_meta.csv",
        "validation_expr.parquet",
        "validation_meta.csv",
    ]
    cache_ready = all(
        os.path.exists(os.path.join(C.RES_DIR, filename))
        for filename in cache_files
    )

    if args.simulate:
        banner("SIMULATION MODE")
        return simulate.run(), True

    if args.resume and cache_ready:
        banner("STAGES 1–2  Resuming from cached harmonized matrices")
        return load_cached_data()

    try:
        banner("STAGE 1  GEO data ingestion")
        datasets = geo_ingest.ingest_all()

        banner("STAGE 2  Preprocessing, harmonization, and batch correction")
        return preprocess.run(datasets), False
    except Exception as error:
        print("\n[WARNING] Live GEO ingestion/preprocessing failed.")
        print(f"          {type(error).__name__}: {str(error)[:300]}")
        print("          Running the clearly labelled simulation fallback.")
        traceback.print_exc(limit=2)
        return simulate.run(), True


def validate_alignment(expr, meta, cohort_name):
    """Ensure expression columns and metadata indices describe identical samples."""
    missing_metadata = set(expr.columns) - set(meta.index)
    if missing_metadata:
        raise ValueError(
            f"{cohort_name} metadata are missing samples: "
            f"{sorted(missing_metadata)[:10]}"
        )
    meta = meta.loc[expr.columns].copy()
    if not expr.columns.equals(meta.index):
        raise ValueError(f"{cohort_name} expression and metadata are not aligned.")
    if "group" not in meta.columns:
        raise ValueError(f"{cohort_name} metadata do not contain a 'group' column.")
    return meta


def main():
    parser = argparse.ArgumentParser(
        description="HepatoSwitch hepatic diabesity transcriptomics pipeline"
    )
    parser.add_argument(
        "--simulate", action="store_true", help="force simulated data"
    )
    parser.add_argument(
        "--resume", action="store_true", help="reuse cached matrices"
    )
    args, _ = parser.parse_known_args()

    banner("HEPATIC DIABESITY TRANSCRIPTOMICS PIPELINE")
    print(f"  Output directory: {C.OUT_DIR}")

    pre, simulated = get_data(args)
    data_mode = "SIMULATED" if simulated else "REAL GEO DATA"
    print(f"\n  >>> DATA MODE: {data_mode} <<<")

    disc_expr = pre["discovery_expr"]
    disc_meta = validate_alignment(
        disc_expr, pre["discovery_meta"], "Discovery cohort"
    )
    val_expr = pre["validation_expr"]
    val_meta = validate_alignment(
        val_expr, pre["validation_meta"], "Validation cohort"
    )
    pre["discovery_meta"] = disc_meta
    pre["validation_meta"] = val_meta

    print(
        f"  Discovery pool: {disc_expr.shape[0]} genes × "
        f"{disc_expr.shape[1]} samples"
    )
    print(f"  Validation cohort: {val_expr.shape[1]} samples")
    print("\nDiscovery cohort composition:")
    print(disc_meta["group"].value_counts().reindex(C.GROUPS, fill_value=0))
    print("\nValidation cohort composition:")
    print(val_meta["group"].value_counts().reindex(C.GROUPS, fill_value=0))

    banner("STAGE 3  Differential expression (3 contrasts)")
    degs = differential_expression.run(disc_expr, disc_meta)
    target_degs = degs.loc[
        degs["contrast"].eq("Lean vs Obese T2D")
    ].copy()
    print(
        f"  Diagnostic: {len(target_degs)} genes tested for Lean vs Obese T2D; "
        f"{int(target_degs['significant'].sum())} meet the configured thresholds."
    )

    banner("STAGE 4  Pathway enrichment (GSEA + ORA)")
    enrichment = pathway_enrichment.run(degs, disc_expr)

    banner("STAGE 5  Molecular-switch prioritization")
    switches = network_switches.run(disc_expr, disc_meta)
    display_columns = [
        column
        for column in ["gene", "present", "spearman_r", "spearman_p", "FDR", "concordant"]
        if column in switches.columns
    ]
    print(switches[display_columns].to_string(index=False))

    banner("STAGE 6  Molecular-staging classifier (LOOCV + validation)")
    ml_results = ml_classifier.run(
        disc_expr, disc_meta, val_expr, val_meta, degs
    )

    banner("STAGE 7  Publication figures and tables")
    figure_paths = figures.run_all(
        pre, degs, enrichment["gsea"], ml_results, simulated
    )

    banner("PIPELINE COMPLETE")
    print(f"  Data mode: {data_mode}")
    print(f"  Results directory: {C.OUT_DIR}")
    print("  Generated figures:")
    for figure_path in figure_paths:
        print(f"    - {figure_path}")


main()
